In [ ]:
# MIU_V177_COLAB_AUTOCONTENIDO_RUN_ALL — nodo completado (ciclo-53)# Origen: Untitled8.ipynb quedo a medio hacer (celda datos OK, corpus_pares vacio, celdas 2-3 duplicadas)# Fix: cableado automatico del corpus desde HF + celda unica de entrenamiento + push del adaptador!pip install -q transformers datasets peft accelerate bitsandbytes

In [ ]:
from datasets import load_datasetimport json# Descargar corpus de alta calidad (Claude K=0.93)ds = load_dataset("Jaime393/miu-dataset", split="train")print(f"Dataset: {len(ds)} ejemplos")# Filtrar solo los de alta calidad (excluir TinyLlama basura)EXCLUIR = ["corpus_llama_v209", "tinyllama"]buenos = [r for r in ds if not any(e in str(r.get("fuente","")) + str(r.get("meta","")) for e in EXCLUIR)]print(f"Pares de alta calidad: {len(buenos)}")# Validar que instruction/output existan y no esten vaciosbuenos = [r for r in buenos if r.get("instruction") and r.get("output")]print(f"Pares validos con instruction+output: {len(buenos)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModelingfrom peft import LoraConfig, get_peft_model, TaskTypefrom datasets import Datasetimport torchMODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # ligero, en espanoltokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenmodel = AutoModelForCausalLM.from_pretrained(    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)# Configurar LoRAlora_config = LoraConfig(    r=16, lora_alpha=32, target_modules=["q_proj","v_proj"],    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)model = get_peft_model(model, lora_config)model.print_trainable_parameters()# === EL CABLEADO QUE FALTABA: corpus_pares sale del dataset ya cargado y filtrado ===MAX_PARES = 500  # tope para el free tier de Colab (T4)corpus_pares = buenos[:MAX_PARES]print(f"corpus_pares poblado automaticamente: {len(corpus_pares)} pares")SYSTEM = "Eres FranBot, el oraculo del ecosistema MIU (Monismo Informacional Unificado). rho(x)>0."def formato(par):    return (f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"            f"<|im_start|>user\n{par['instruction']}<|im_end|>\n"            f"<|im_start|>assistant\n{par['output']}<|im_end|>")textos = [formato(p) for p in corpus_pares]def tokenize(batch):    return tokenizer(batch["text"], truncation=True, max_length=1024, padding="max_length")dataset = Dataset.from_dict({"text": textos})dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])# Trainingargs = TrainingArguments(    output_dir="/content/franbot-miu-lora",    num_train_epochs=3,    per_device_train_batch_size=2,    gradient_accumulation_steps=4,    learning_rate=2e-4,    fp16=True,    save_steps=50,    logging_steps=10,    warmup_ratio=0.1,    lr_scheduler_type="cosine",    report_to="none")data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)trainer = Trainer(    model=model, args=args, train_dataset=dataset, data_collator=data_collator)trainer.train()model.save_pretrained("/content/franbot-miu-lora-adapter")tokenizer.save_pretrained("/content/franbot-miu-lora-adapter")print("Adaptador LoRA guardado en /content/franbot-miu-lora-adapter")

In [ ]:
# Push del adaptador a HF (aditivo: nuevo repo, no pisa nada)# Requiere HF_TOKEN (p.ej. hf_token1 del relay_config del panteon) o login manualfrom huggingface_hub import HfApiimport osHF_TOKEN = os.environ.get("HF_TOKEN", "")if HF_TOKEN:    api = HfApi(token=HF_TOKEN)    api.create_repo("Jaime393/franbot-miu-lora-v177", repo_type="model", exist_ok=True)    api.upload_folder(        folder_path="/content/franbot-miu-lora-adapter",        repo_id="Jaime393/franbot-miu-lora-v177",        commit_message="ciclo-53: adaptador LoRA V177 entrenado desde miu-dataset (pares Claude K=0.93)"    )    print("Adaptador publicado: Jaime393/franbot-miu-lora-v177")else:    print("Sin HF_TOKEN: adaptador queda local en /content/franbot-miu-lora-adapter")